# Where does revenue come from, by customer type and channel, and does it change the branch?

**Week 1, Monday, afternoon: the second case, in pairs, 25 minutes.** Kalpa Retail's revenue grew
4 percent last year against a growth plan of 15 percent, and marketing wants Rs 12 crore to win
new customers. Meera asked two things at the start of the day, and the chapters answered the
second, which branch of sales is short.
This case answers the first on the same 30 orders, placed from 1 July to 26 September 2026, and
tests whether the channel view changes the recommendation.

> Meera Raghavan, CEO of Kalpa Retail: "Where does revenue come from, by customer type and
> channel? Is acquisition even the branch that is short?"
>
> Anand Iyer, the finance controller: "No averages. One business customer can move an average."

**Who needs the answer.** Meera will open the channel slide before she reads the sentence, and
the store team says they carry the business. A share read without the orders behind it can turn
the growth plan store-led, and the Rs 12 crore would follow the share.

**The questions on the way.**

1. What share of booked revenue does each channel bring?
2. How many orders stand behind each channel's share, and what are they like?
3. Which orders does Meera's growth plan concern, and what share of them does each channel hold?
4. Where does consumer revenue leak between booked and delivered, channel by channel?
5. How much consumer revenue does each customer type bring?
6. Does the channel view change the branch Meera opens first?

The metric at stake is share of revenue: a channel's or a customer type's rupees over the total
they are part of. Booked revenue counts every order placed, cancellations and returns included,
and delivered counts what reached a customer and stayed. The chapters found frequency, the
orders-per-customer branch, the one to open first, since on the 30 booked orders 16 of 23
customers bought once and 7 came back, and
the escalated case rebuilt that answer on delivered orders.

> **Kavya's review.** Meera will open the channel slide before she reads your sentence, and the
> first thing on it will be store at nine rupees in ten. Decide whether that number changes your
> answer before she asks, and count the orders behind every share you show her.

Each step carries a `TODO` marker. Above each `__TODOn__` placeholder is a lettered choice;
replace the placeholder with the option you pick, run the cell, then run the check under it.
Run from the top: the notebook stops at the first placeholder with a `NameError` naming it,
which is intended.

**Setup.** The first cell finds the shared helper, loads the 30 orders from `../data/`, turns
every amount into a whole number with `int()`, the fix chapter 1 found for an amount stored as
text, and sums booked revenue for the shares below.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

ORDERS = kit.load_records()

import statistics
for order in ORDERS:
    order["amount"] = int(order["amount"])     # chapter 1's fix for an amount stored as text
booked_revenue = 0
for order in ORDERS:
    booked_revenue += order["amount"]
channels = ["app", "web", "store"]
print(len(ORDERS), "orders loaded, every amount a whole number; booked revenue", kit.rupees(booked_revenue))

In [ ]:
kit.side_by_side(
    kit.ladder(["The chapters: is acquisition short?", "The escalated case: does it survive?",
                "The second case: does channel change it?"], lit=2, show=False),
    kit.vflow(["1. What share does each channel bring?", "2. What orders stand behind it?",
               "3. Which orders does the plan concern?", "4. Where does revenue leak?",
               "5. What does each customer type bring?", "6. Does the branch change?"], show=False),
)

## Step 1. What share of booked revenue does each channel bring?

A channel mix review opens most retail operating meetings, and a share is the first number on
its slide. Add each order's amount to its channel's total, then work out store's share of booked
revenue.

**Write down:** each channel's share of booked revenue, and store's in particular.

In [ ]:
channel_revenue = {}
channel_orders = {}
for order in ORDERS:
    ch = order["channel"]
    # TODO 1. Which line adds this order's amount to its channel's total?
    #   a) channel_revenue[ch] = order["amount"]
    #   b) channel_revenue[ch] = channel_revenue.get(ch, 0) + 1
    #   c) channel_revenue[ch] = channel_revenue.get(ch, 0) + order["amount"]
    #   d) channel_revenue[ch] = channel_revenue.get(ch, order["amount"]) + order["amount"]
    __TODO1__
    channel_orders[ch] = channel_orders.get(ch, 0) + 1

# TODO 2. Which expression is store's share of booked revenue?
#   a) channel_revenue["store"] / len(ORDERS)
#   b) channel_revenue["store"] / booked_revenue
#   c) channel_orders["store"] / len(ORDERS)
#   d) channel_revenue["store"] / channel_revenue["app"]
store_share = __TODO2__

total = sum(channel_revenue.values())
kit.table(["Channel", "Orders", "Share of booked revenue"],
          [(ch, channel_orders[ch], f"{channel_revenue[ch] / total * 100:.1f}%") for ch in channels],
          caption="Share of booked revenue by channel, all 30 orders")
kit.bars([(ch, round(channel_revenue[ch] / total * 1000) / 10) for ch in channels],
         fmt=lambda v: f"{v:.1f}%", lit=(2,), title="Share of booked revenue by channel, percent")

In [ ]:
kit.check("the channels reconcile to booked revenue", sum(channel_revenue.values()) == booked_revenue)
kit.check("each channel carries the same number of orders", len(set(channel_orders.values())) == 1,
          str(channel_orders))
kit.check("store's share of rupees sits above its share of orders and below one",
          channel_orders["store"] / len(ORDERS) < store_share < 1, f"{store_share * 100:.1f} percent")

## Step 2. How many orders stand behind each channel's share, and what are they like?

Sales operations teams count the orders behind a share before anyone plans around it, and they
set a channel's average order beside its typical one. Count each channel's orders by status, then
set each channel's mean order beside its median.

**Write down:** each channel's delivered, returned and cancelled counts, and each channel's mean
and median order.

In [ ]:
status_count = {"app": {}, "web": {}, "store": {}}
for order in ORDERS:
    ch = order["channel"]
    st = order["status"]
    # TODO 3. Which line counts this order under its channel and its status?
    #   a) status_count[ch] = status_count.get(ch, 0) + 1
    #   b) status_count[st][ch] = status_count[st].get(ch, 0) + 1
    #   c) status_count[ch][st] = status_count[ch].get(st, 0) + order["amount"]
    #   d) status_count[ch][st] = status_count[ch].get(st, 0) + 1
    __TODO3__

channel_mean = {ch: channel_revenue[ch] / channel_orders[ch] for ch in channels}
channel_median = {ch: statistics.median([o["amount"] for o in ORDERS if o["channel"] == ch])
                  for ch in channels}
statuses = ["delivered", "returned", "cancelled"]
kit.table(["Channel", "Delivered", "Returned", "Cancelled", "Mean order", "Median order"],
          [(ch, *[status_count[ch].get(st, 0) for st in statuses], kit.rupees(round(channel_mean[ch])),
            kit.rupees(round(channel_median[ch]))) for ch in channels],
          caption="The orders behind each channel's share")
kit.columns(channels, [(st, [status_count[ch].get(st, 0) for ch in channels]) for st in statuses],
            title="Each channel's 10 orders by status", fmt=lambda v: f"{v:.0f}")

In [ ]:
kit.check("the status counts add back to 30 orders",
          sum(sum(c.values()) for c in status_count.values()) == len(ORDERS))
kit.check("every returned order came through one channel",
          sum(1 for ch in channels if status_count[ch].get("returned", 0) > 0) == 1)
kit.check("store's mean order is more than ten times its median",
          channel_mean["store"] > 10 * channel_median["store"],
          f"{kit.rupees(round(channel_mean['store']))} against {kit.rupees(round(channel_median['store']))}")

## Step 3. Which orders does Meera's growth plan concern, and what share of them does each channel hold?

An analyst scopes a number to the customers a decision concerns before comparing channels.
Meera's growth plan concerns Kalpa's three consumer segments, Retail-Core, Retail-Plus and
Student, since those are the customers a retention offer or an acquisition campaign reaches. The
consumer view keeps the orders whose segment is one of those three; recompute the channel view on
it.

**Write down:** consumer booked revenue, and each channel's consumer revenue and share.

In [ ]:
CONSUMER = ("Retail-Core", "Retail-Plus", "Student")     # the segments Meera's plan concerns
# TODO 4. Which condition keeps the orders in the three consumer segments Meera's plan concerns?
#   a) order["segment"] in CONSUMER
#   b) order["segment"] == "Retail-Core"
#   c) order["channel"] != "store"
#   d) order["status"] != "cancelled"
consumer = []
for order in ORDERS:
    if __TODO4__:
        consumer.append(order)

consumer_revenue = 0
consumer_channel_rev = {"app": 0, "web": 0, "store": 0}
for order in consumer:
    consumer_revenue += order["amount"]
    consumer_channel_rev[order["channel"]] += order["amount"]

kit.table(["Channel", "Consumer revenue", "Share of consumer revenue"],
          [(ch, kit.rupees(consumer_channel_rev[ch]),
            f"{consumer_channel_rev[ch] / consumer_revenue * 100:.1f}%") for ch in channels],
          caption=f"The consumer view: {kit.rupees(consumer_revenue)} booked")
kit.columns(channels, [("share of all booked revenue", [channel_revenue[ch] / booked_revenue * 100 for ch in channels]),
                       ("share of consumer revenue", [consumer_channel_rev[ch] / consumer_revenue * 100 for ch in channels])],
            title="Each channel's share before and after the consumer view, in percent", fmt=lambda v: f"{v:.1f}")

In [ ]:
kit.check("the consumer view keeps every order of the three consumer segments and no other",
          {o["segment"] for o in consumer} <= set(CONSUMER)
          and all(sum(o["segment"] == s for o in consumer) == sum(o["segment"] == s for o in ORDERS)
                  for s in CONSUMER))
kit.check("the consumer channels reconcile to consumer revenue",
          sum(consumer_channel_rev.values()) == consumer_revenue, kit.rupees(consumer_revenue))
kit.check("on the consumer view store no longer leads",
          consumer_channel_rev["store"] < max(consumer_channel_rev.values()))

## Step 4. Where does consumer revenue leak between booked and delivered, channel by channel?

A returns rate and a cancellation rate by channel are standard lines in an e-commerce operating
review. Booked is what customers asked for and delivered is what stayed sold, so split each
channel's consumer revenue by status and find where it leaks.

**Write down:** web's returned revenue, store's cancelled revenue, and consumer delivered revenue.

In [ ]:
consumer_status_rev = {"app": {}, "web": {}, "store": {}}
for order in consumer:
    ch = order["channel"]
    st = order["status"]
    consumer_status_rev[ch][st] = consumer_status_rev[ch].get(st, 0) + order["amount"]

# TODO 5. Which number is web's leak, the booked revenue that came back?
#   a) consumer_status_rev["web"].get("delivered", 0)
#   b) consumer_status_rev["web"].get("returned", 0)
#   c) consumer_status_rev["store"].get("returned", 0)
#   d) consumer_channel_rev["web"] - consumer_status_rev["web"].get("returned", 0)
web_leak = __TODO5__
store_leak = consumer_status_rev["store"].get("cancelled", 0)
consumer_delivered = 0
for ch in channels:
    consumer_delivered += consumer_status_rev[ch].get("delivered", 0)

kit.table(["Channel", "Delivered", "Returned", "Cancelled"],
          [(ch, *[kit.rupees(consumer_status_rev[ch].get(st, 0)) for st in statuses]) for ch in channels],
          caption="Consumer revenue by channel and status")
kit.columns(channels, [("booked", [consumer_channel_rev[ch] for ch in channels]),
                       ("delivered", [consumer_status_rev[ch].get("delivered", 0) for ch in channels])],
            title="Consumer revenue by channel, booked against delivered", fmt=kit.rupees)
kit.bridge(("consumer booked", consumer_revenue),
           [("web returns", -web_leak), ("store cancellations", -store_leak)],
           end_label="consumer delivered", lit=(0, 1),
           title="Where consumer revenue leaks between booked and delivered")

In [ ]:
kit.check("booked less the two leaks lands on delivered",
          consumer_revenue - web_leak - store_leak == consumer_delivered, kit.rupees(consumer_delivered))
kit.check("app delivered every rupee it booked",
          consumer_status_rev["app"].get("delivered", 0) == consumer_channel_rev["app"])
kit.check("web lost more than a third of its booked revenue to returns",
          web_leak > consumer_channel_rev["web"] / 3, kit.rupees(web_leak))

## Step 5. How much consumer revenue does each customer type bring?

Segment reporting is how the head of a membership tier sees what the tier earns. Meera asked
about customer type, and the segment is recorded on each order, so this step stays at revenue by
segment: one repeat customer appears once as Retail-Core and once as Retail-Plus, and a count of
customers per segment would count that customer twice.

**Write down:** each consumer segment's revenue, its share of consumer revenue and its revenue
per order.

In [ ]:
segment_rev, segment_delivered, segment_orders = {}, {}, {}
for order in consumer:
    seg = order["segment"]
    segment_rev[seg] = segment_rev.get(seg, 0) + order["amount"]
    segment_orders[seg] = segment_orders.get(seg, 0) + 1
    if order["status"] == "delivered":
        segment_delivered[seg] = segment_delivered.get(seg, 0) + order["amount"]

# TODO 6. Which total is the denominator for a segment's share of consumer revenue?
#   a) booked_revenue
#   b) len(consumer)
#   c) consumer_revenue
#   d) segment_rev["Retail-Core"]
denominator = __TODO6__

segments = list(CONSUMER)
kit.table(["Customer type", "Booked revenue", "Share of consumer revenue", "Revenue per order", "Delivered revenue"],
          [(seg, kit.rupees(segment_rev[seg]), f"{segment_rev[seg] / denominator * 100:.1f}%",
            kit.rupees(round(segment_rev[seg] / segment_orders[seg])), kit.rupees(segment_delivered.get(seg, 0)))
           for seg in segments],
          caption="Consumer revenue by customer type, recorded on each order")
kit.columns(segments, [("booked", [segment_rev[seg] for seg in segments]),
                       ("delivered", [segment_delivered.get(seg, 0) for seg in segments])],
            title="Consumer revenue by customer type, booked against delivered", fmt=kit.rupees)

In [ ]:
kit.check("the three consumer segments reconcile to consumer revenue",
          sum(segment_rev.values()) == consumer_revenue, kit.rupees(sum(segment_rev.values())))
kit.check("the shares of consumer revenue add to one",
          abs(sum(segment_rev[s] / denominator for s in segments) - 1) < 1e-9)
kit.check("each customer type delivered no more than it booked",
          all(segment_delivered.get(s, 0) <= segment_rev[s] for s in segments))

## Step 6. Does the channel view change the branch Meera opens first?

A recommendation is tested against each new cut of the data before it reaches the board. The
chapters and the escalated case said open frequency first; weigh that against what the channel
view found.

In [ ]:
# TODO 7. What does the channel view do to the recommendation?
#   a) "frequency first, two leaks named"
#   b) "store-led, since store brings 91.6 percent"
#   c) "web-led"
#   d) "acquisition first"
verdict = __TODO7__
print("the recommendation:", verdict)
kit.flow([f"store: {store_share * 100:.1f} percent of all booked rupees",
          f"store's mean order: {channel_mean['store'] / channel_median['store']:.0f} times its median",
          f"consumer view: store {consumer_channel_rev['store'] / consumer_revenue * 100:.1f} percent",
          "web returns, store cancellations", "the recommendation: " + verdict],
         kinds=["bad", "unknown", "known", "known", "lit"],
         title="From store's headline share to the recommendation")

In [ ]:
kit.check("no channel holds half of consumer revenue", max(consumer_channel_rev.values()) < consumer_revenue / 2)
kit.check("both leaks carry rupees", web_leak > 0 and store_leak > 0,
          f"web returns {kit.rupees(web_leak)}, store cancellations {kit.rupees(store_leak)}")

### What will an interviewer ask about channel shares, and how do you answer?

Answer each aloud in the drill, with this notebook's numbers, then compare with the solution.

- **[D] One channel carries nine rupees in ten of revenue; does that change where the growth
  plan invests?**
- **[F] A business says "grow revenue 15 percent"; how do you turn that into questions data can
  answer?**

In [ ]:
kit.table(["Step", "What it established"], [
    ("1. by channel", f"store carries {store_share * 100:.1f} percent of booked revenue from a third of the orders"),
    ("2. the orders behind it", f"store's mean order is {kit.rupees(round(channel_mean['store']))} against a median "
                                f"of {kit.rupees(round(channel_median['store']))}"),
    ("3. the consumer view", f"store holds {kit.rupees(consumer_channel_rev['store'])} of "
                             f"{kit.rupees(consumer_revenue)}, "
                             f"{consumer_channel_rev['store'] / consumer_revenue * 100:.1f} percent"),
    ("4. by status", f"web returned {kit.rupees(web_leak)}; store cancelled {kit.rupees(store_leak)}; "
                     f"app delivered all {kit.rupees(consumer_channel_rev['app'])}"),
    ("5. by customer type", f"Retail-Core {kit.rupees(segment_rev['Retail-Core'])}, Retail-Plus "
                            f"{kit.rupees(segment_rev['Retail-Plus'])}, Student {kit.rupees(segment_rev['Student'])}"),
    ("6. the recommendation", verdict)], caption="What the second case established")
kit.bars([(ch, consumer_status_rev[ch].get("delivered", 0)) for ch in channels], fmt=kit.rupees,
         title="What stayed sold: consumer delivered revenue by channel")

## What does the pair post when every step has run?

One post per pair: the brief's line of six letters, then this notebook's seven TODO picks in
order as a second line, then one sentence answering Meera's channel question. Every check should
print PASS before you post.

In [ ]:
kit.check_summary()
print("Next: Tuesday puts the quarter before this one beside it, to see which branch moved.")